In [18]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.neighbors import LocalOutlierFactor
import pandas as pd

In [19]:
def show_view(df, view_cols):
    """Render a result table with the viewpoint's own attributes highlighted.

    Highlighted = the attributes this viewpoint is allowed to observe, i.e. the ones
    that produced the LOF score. Greyed out = present in the data but hidden from
    this detector, so they had no influence on the verdict.
    """
    hidden = [c for c in df.columns if c not in view_cols + ["type", "LOF"]]
    return (
        df.style
          .set_properties(subset=view_cols,
                          **{"background-color": "#fde68a", "color": "#111111"})
          .set_properties(subset=hidden, **{"color": "#9ca3af"})
          .set_properties(subset=["LOF"], **{"font-weight": "bold"})
          .format(precision=2)
          .set_caption("highlighted = observed by this viewpoint &nbsp;|&nbsp; grey = hidden from it")
    )

## Viewpoint 1 — Geographic location

The first observability point sees only **2 of the 8 attributes**: `Latitude` and `Longitude`.
All 20,640 census block groups are still present — we restrict *what* is observed, not *who*.

Under this viewpoint "normal" means **surrounded by neighbours at a similar density**, so an anomaly is a
block group that is geographically isolated relative to its surroundings. Local Outlier Factor is used
because population density in California varies enormously: a plain distance measure would flag every
rural block simply for being rural, whereas LOF compares each block against the density of its *own*
neighbourhood.

LOF reads as a ratio — `1.0` = as dense as its neighbours, `> 1.0` = sparser (isolated), `< 1.0` = denser.
The table below contrasts the 5 most isolated block groups with the 5 most typical ones, showing **all**
attributes so we can confirm that the other six say nothing unusual about them.


In [20]:
housing = fetch_california_housing()
X_all = housing.data
names = list(housing.feature_names)

# the LOCATION view: 2 attributes, so we can plot it directly
loc_cols = ["Latitude", "Longitude"]
loc = [names.index(c) for c in loc_cols]
X = X_all[:, loc]
# print(X.shape)   # (20640, 2)

clf = LocalOutlierFactor(n_neighbors=20)
clf.fit_predict(X)
scores = clf.negative_outlier_factor_
lof = -scores          # flip to positive: 1.0 = normal, higher = more isolated

pd.set_option("display.max_columns", None)      # stop pandas truncating wide tables
pd.set_option("display.width", 200)

top5     = np.argsort(scores)[:5]                    # most anomalous
typical5 = np.argsort(np.abs(lof - 1.0))[:5]         # most typical: LOF closest to 1.0

rows = np.concatenate([top5, typical5])

df = pd.DataFrame(X_all[rows], columns=names).round(2)
df.insert(0, "LOF", lof[rows].round(2))
df.insert(0, "type", ["anomaly"] * 5 + ["typical"] * 5)
df.index = rows
df.index.name = "block"
show_view(df, loc_cols)

,type,LOF,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
block,,,,,,,,,,
8316,anomaly,13.67,3.39,52.00,7.39,1.78,733.00,2.55,33.33,-118.32
8315,anomaly,13.41,2.83,52.00,5.47,1.37,1100.00,2.55,33.34,-118.33
8317,anomaly,13.40,2.74,52.00,6.22,1.65,341.00,2.13,33.34,-118.32
8314,anomaly,13.13,2.16,27.00,5.06,1.57,744.00,2.25,33.35,-118.32
2799,anomaly,10.54,2.10,19.00,3.77,1.46,490.00,2.99,36.40,-117.02
4724,typical,1.00,5.09,52.00,5.45,1.03,832.00,2.02,34.06,-118.37
15772,typical,1.00,4.37,52.00,4.91,1.04,1374.00,1.92,37.77,-122.44
15711,typical,1.00,4.09,52.00,4.31,1.02,1224.00,2.53,37.79,-122.44
15710,typical,1.00,4.45,52.00,4.39,1.06,831.00,1.92,37.79,-122.44


## Viewpoint 2 — Dwelling (the housing stock)

The second observability point sees a different **3 of the 8** attributes: `HouseAge`, `AveRooms`, `AveBedrms`.
The same 20,640 block groups are present — only the columns change.

Under this viewpoint "normal" means a **plausible combination of building age, size and layout**. A block is
anomalous if its housing stock does not hold together as a description of real dwellings.

Two preparation steps that viewpoint 1 did not need:

- **`log1p` on `AveRooms` and `AveBedrms`.** Both are ratios (rooms ÷ households) with extreme tails —
  `AveRooms` reaches 141.9 against a 99th percentile of 10.4. Left raw, a handful of blocks would define
  the neighbourhood structure for every other block.
- **Standardisation.** The three attributes use different units (years vs. rooms), so without rescaling
  `HouseAge` would dominate every distance simply because its numbers are larger. Viewpoint 1 needed
  neither step because latitude and longitude are already on a common scale and are not skewed.

`n_neighbors` stays at **20** for every viewpoint. If it were tuned per view, differences between the views
would no longer be about the views.


In [21]:
from sklearn.preprocessing import StandardScaler

# the DWELLING view: what the housing stock looks like
dwl_cols = ["HouseAge", "AveRooms", "AveBedrms"]
X_dwl = X_all[:, [names.index(c) for c in dwl_cols]].copy()

X_dwl[:, 1] = np.log1p(X_dwl[:, 1])          # AveRooms   - heavy tail
X_dwl[:, 2] = np.log1p(X_dwl[:, 2])          # AveBedrms  - heavy tail
X_dwl = StandardScaler().fit_transform(X_dwl)

clf_dwl = LocalOutlierFactor(n_neighbors=20)  # same k as viewpoint 1
clf_dwl.fit_predict(X_dwl)
lof_dwl = -clf_dwl.negative_outlier_factor_

top5_dwl     = np.argsort(-lof_dwl)[:5]                  # most anomalous
typical5_dwl = np.argsort(np.abs(lof_dwl - 1.0))[:5]     # most typical
rows_dwl     = np.concatenate([top5_dwl, typical5_dwl])

df_dwl = pd.DataFrame(X_all[rows_dwl], columns=names).round(2)
df_dwl.insert(0, "LOF", lof_dwl[rows_dwl].round(2))
df_dwl.insert(0, "type", ["anomaly"] * 5 + ["typical"] * 5)
df_dwl.index = rows_dwl
df_dwl.index.name = "block"
show_view(df_dwl, dwl_cols)

,type,LOF,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
block,,,,,,,,,,
1979,anomaly,3.42,4.62,34.00,132.53,34.07,36.00,2.40,38.80,-120.08
19435,anomaly,3.36,0.54,16.00,2.11,2.11,166.00,18.44,37.67,-121.04
4559,anomaly,3.27,4.10,52.00,2.15,1.93,41.00,1.52,34.05,-118.26
1914,anomaly,2.91,1.88,33.00,141.91,25.64,30.00,2.73,38.91,-120.10
3126,anomaly,2.70,2.38,4.00,1.00,1.00,6.00,3.00,35.21,-117.79
19949,typical,1.00,1.45,35.00,4.96,1.06,1567.00,3.49,36.21,-119.37
10383,typical,1.00,3.41,16.00,5.41,1.05,1202.00,1.64,33.62,-117.64
6497,typical,1.00,2.33,31.00,3.98,1.05,2830.00,4.69,34.08,-118.02
20047,typical,1.00,1.38,27.00,4.08,1.07,839.00,3.04,36.05,-119.01


## Viewpoint 3 — People (who lives there)

The third observability point sees the remaining **3 of the 8** attributes: `MedInc`, `Population`, `AveOccup`.

Under this viewpoint "normal" means a **plausible community profile** — an income level, a headcount and a
household size that fit together. A block is anomalous if that profile does not describe a real population.

`Population` and `AveOccup` receive the same `log1p` treatment for the same reason as viewpoint 2:
`AveOccup` reaches 1243 people per household against a 99th percentile of 5.4. `MedInc` is already
well behaved (0.5 to 15) and is left alone apart from standardisation.


In [22]:
# the PEOPLE view: who lives there
ppl_cols = ["MedInc", "Population", "AveOccup"]
X_ppl = X_all[:, [names.index(c) for c in ppl_cols]].copy()

X_ppl[:, 1] = np.log1p(X_ppl[:, 1])          # Population - heavy tail
X_ppl[:, 2] = np.log1p(X_ppl[:, 2])          # AveOccup   - heavy tail
X_ppl = StandardScaler().fit_transform(X_ppl)

clf_ppl = LocalOutlierFactor(n_neighbors=20)  # same k as viewpoints 1 and 2
clf_ppl.fit_predict(X_ppl)
lof_ppl = -clf_ppl.negative_outlier_factor_

top5_ppl     = np.argsort(-lof_ppl)[:5]                  # most anomalous
typical5_ppl = np.argsort(np.abs(lof_ppl - 1.0))[:5]     # most typical
rows_ppl     = np.concatenate([top5_ppl, typical5_ppl])

df_ppl = pd.DataFrame(X_all[rows_ppl], columns=names).round(2)
df_ppl.insert(0, "LOF", lof_ppl[rows_ppl].round(2))
df_ppl.insert(0, "type", ["anomaly"] * 5 + ["typical"] * 5)
df_ppl.index = rows_ppl
df_ppl.index.name = "block"
show_view(df_ppl, ppl_cols)

,type,LOF,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
block,,,,,,,,,,
19006,anomaly,5.23,10.23,45.00,3.17,0.83,7460.00,1243.33,38.32,-121.98
3364,anomaly,4.36,5.52,36.00,5.14,1.14,4198.00,599.71,40.41,-120.51
6399,anomaly,4.27,15.00,35.00,8.59,1.07,268.00,9.24,34.13,-118.04
16669,anomaly,4.19,4.26,46.00,9.08,1.31,6532.00,502.46,35.32,-120.70
13034,anomaly,3.46,6.14,52.00,8.28,1.52,6675.00,230.17,38.69,-121.15
3238,typical,1.00,2.13,25.00,4.34,1.04,893.00,3.54,36.10,-119.56
1528,typical,1.00,5.56,38.00,6.29,1.01,809.00,2.38,37.89,-122.07
5899,typical,1.00,3.95,37.00,4.93,1.03,860.00,1.98,34.16,-118.31
12986,typical,1.00,4.20,23.00,6.15,0.97,1022.00,2.93,38.67,-121.30


## The joint view — no viewpoint restriction

This is the **control**, not a fourth viewpoint. The detector sees **all 8 attributes** at once, which is what
you would do if the multi-viewpoint idea had never been introduced — so it is the thing any viewpoint-based
method has to be compared against.

The preprocessing (`log1p` on the heavy-tailed attributes, then standardisation), the `n_neighbors = 20`
setting and the table layout are all identical to viewpoints 1–3. The **only** thing that differs is which
columns are visible, so any difference in the results is attributable to observability and nothing else.
Every column is highlighted below, because nothing is hidden from this detector.

Under this view "normal" means a **plausible complete description of a block group** — location, housing stock
and population all considered together.

Watch what happens to the isolated blocks from viewpoint 1. Catalina Island ranks **1st of 20,640** when the
detector sees only latitude and longitude, and **287th** when it sees all eight. Nothing about the block
changed; its six ordinary attributes simply outvoted its two extraordinary ones. That signal loss is the
"fusion dilutes view-specific signals" effect, and it is why per-view detectors remain useful even where the
joint detector scores better overall.


In [26]:
# the JOINT view: all 8 attributes at once -- no viewpoint restriction at all
all_cols = names                 # every column (this dataset has no ID column to drop)

X_joint = X_all.copy()
for c in ["AveRooms", "AveBedrms", "Population", "AveOccup"]:
    X_joint[:, names.index(c)] = np.log1p(X_joint[:, names.index(c)])   # heavy tails
X_joint = StandardScaler().fit_transform(X_joint)

clf_all = LocalOutlierFactor(n_neighbors=20)   # same k as the three viewpoints
clf_all.fit_predict(X_joint)
lof_all = -clf_all.negative_outlier_factor_

top5_all     = np.argsort(-lof_all)[:5]                  # most anomalous
typical5_all = np.argsort(np.abs(lof_all - 1.0))[:5]     # most typical
rows_all     = np.concatenate([top5_all, typical5_all])

df_all = pd.DataFrame(X_all[rows_all], columns=names).round(2)
df_all.insert(0, "LOF", lof_all[rows_all].round(2))
df_all.insert(0, "type", ["anomaly"] * 5 + ["typical"] * 5)
df_all.index = rows_all
df_all.index.name = "block"
show_view(df_all, all_cols)

,type,LOF,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
block,,,,,,,,,,
19006,anomaly,3.75,10.23,45.00,3.17,0.83,7460.00,1243.33,38.32,-121.98
3364,anomaly,3.18,5.52,36.00,5.14,1.14,4198.00,599.71,40.41,-120.51
16669,anomaly,3.16,4.26,46.00,9.08,1.31,6532.00,502.46,35.32,-120.70
6399,anomaly,3.02,15.00,35.00,8.59,1.07,268.00,9.24,34.13,-118.04
14322,anomaly,3.01,1.29,52.00,2.76,0.89,1283.00,7.73,32.70,-117.15
8365,typical,1.00,3.27,26.00,4.27,1.07,1130.00,2.80,33.97,-118.35
4041,typical,1.00,7.14,34.00,7.09,1.05,1187.00,2.37,34.16,-118.50
13951,typical,1.00,5.26,17.00,10.32,1.85,673.00,2.46,34.24,-117.13
6373,typical,1.00,3.88,44.00,5.60,1.01,1045.00,2.42,34.15,-118.02
